
# Yelp Dataset — Exploration + Parquet/DuckDB

Tento notebook:
- převádí JSONL → **Parquet** (particionovaně) pomocí **DuckDB**,
- dělá rychlé **ad‑hoc SQL** dotazy nad Parquetem,
- sestaví **interactions.parquet** (user_id, business_id, ts, stars, implicit=1),
- ukáže **časový split** pro trénink/validaci,
- nechá skeleton pro ELSA + SAE (budeme napojovat podle tvé volby knihoven).


In [3]:

from pathlib import Path
import duckdb, pandas as pd
import numpy as np
import json, os, itertools
from datetime import datetime

# ---- Cesty k datům ----
DATA_DIR = Path("./yelp_dataset")          # změň dle sebe
PARQUET_DIR = Path("./yelp_parquet")
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
con = duckdb.connect("yelp.duckdb")

FILES = {
    "business": DATA_DIR / "yelp_academic_dataset_business.json",
    "review":   DATA_DIR / "yelp_academic_dataset_review.json",
    "user":     DATA_DIR / "yelp_academic_dataset_user.json",
    "tip":      DATA_DIR / "yelp_academic_dataset_tip.json",
    "checkin":  DATA_DIR / "yelp_academic_dataset_checkin.json",
}
FILES


{'business': WindowsPath('yelp_dataset/yelp_academic_dataset_business.json'),
 'review': WindowsPath('yelp_dataset/yelp_academic_dataset_review.json'),
 'user': WindowsPath('yelp_dataset/yelp_academic_dataset_user.json'),
 'tip': WindowsPath('yelp_dataset/yelp_academic_dataset_tip.json'),
 'checkin': WindowsPath('yelp_dataset/yelp_academic_dataset_checkin.json')}

## 1) JSON → Parquet pomocí DuckDB

In [ ]:

# import shutil

# for sub in ["business", "review", "user"]:
#     out_dir = PARQUET_DIR / sub
#     if out_dir.exists():
#         print("Removing", out_dir)
#         shutil.rmtree(out_dir)

# # Business (partitions by state)
# if FILES["business"].exists():
#     con.execute(f"""
#     COPY (
#       SELECT * FROM read_json_auto('{FILES["business"]}', format='auto')
#     ) TO '{PARQUET_DIR/'business'}' (
#       FORMAT PARQUET, COMPRESSION ZSTD, PARTITION_BY (state)
#     );
#     """)
#     print("business → parquet OK")
# else:
#     print("business.json nenalezen")

# # Review (partitions by year)
# if FILES["review"].exists():
#     con.execute(f"""
#     COPY (
#       SELECT *, CAST(strftime(date, '%Y') AS INTEGER) AS year
#       FROM read_json_auto('{FILES["review"]}', format='auto')
#     ) TO '{PARQUET_DIR/'review'}' (
#       FORMAT PARQUET, COMPRESSION ZSTD, PARTITION_BY (year)
#     );
#     """)
#     print("review → parquet OK")
# else:
#     print("review.json nenalezen")

# # User (no partitions, můžeš přidat podle yelping_since)
# if FILES["user"].exists():
#     con.execute(f"""
#     COPY (
#       SELECT * FROM read_json_auto('{FILES["user"]}', format='auto')
#     ) TO '{PARQUET_DIR/'user.parquet'}' (
#       FORMAT PARQUET, COMPRESSION ZSTD
#     );
#     """)
#     print("user → parquet OK")
# else:
#     print("user.json nenalezen")

# # Tip (volitelné)
# if FILES["tip"].exists():
#     con.execute(f"""
#     COPY (
#       SELECT * FROM read_json_auto('{FILES["tip"]}', format='auto')
#     ) TO '{PARQUET_DIR/'tip.parquet'}' (
#       FORMAT PARQUET, COMPRESSION ZSTD
#     );
#     """)
#     print("tip → parquet OK")

# # Checkin (volitelné – může se hodit na implicitní interakce)
# if FILES["checkin"].exists():
#     con.execute(f"""
#     COPY (
#       SELECT * FROM read_json_auto('{FILES["checkin"]}', format='auto')
#     ) TO '{PARQUET_DIR/'checkin.parquet'}' (
#       FORMAT PARQUET, COMPRESSION ZSTD
#     );
#     """)
#     print("checkin → parquet OK")


Removing yelp_parquet\business
Removing yelp_parquet\review
business → parquet OK
review → parquet OK
user → parquet OK
tip → parquet OK
checkin → parquet OK


## 2) Rychlé SQL nad Parquetem (bez loadu do RAM)

In [1]:

# Příklad: top města podle počtu podniků a průměrných hvězd
q = con.execute(f"""
SELECT city, state, COUNT(*) AS n, AVG(CAST(stars AS DOUBLE)) AS avg_stars
FROM read_parquet('{PARQUET_DIR/'business/*/*.parquet'}')
GROUP BY city, state
ORDER BY n DESC
LIMIT 20
""").fetchdf()
q


NameError: name 'con' is not defined

## 3) Vytvoření interactions.parquet (implicit = 1 z reviews)

In [6]:

# Z recenzí vytvoříme interakce (user_id, business_id, timestamp, stars, implicit=1)
# Data se čtou přímo z Parquetu, můžeš přidat filtry (rok, stát) pro menší subsety.
interactions_path = PARQUET_DIR / "interactions.parquet"

con.execute(f"""
COPY (
  SELECT
    user_id,
    business_id,
    TRY_CAST(stars AS DOUBLE) AS stars,
    TRY_CAST(epoch_ms(date) AS BIGINT) AS ts,
    1 AS implicit
  FROM read_parquet('{PARQUET_DIR/'review/year=*/data-*.parquet'}')
  WHERE user_id IS NOT NULL AND business_id IS NOT NULL
) TO '{interactions_path}' (FORMAT PARQUET, COMPRESSION ZSTD);
""")
print("interactions.parquet vytvořen:", interactions_path.exists())


IOException: IO Error: No files found that match the pattern "yelp_parquet\review\year=*\data-*.parquet"

LINE 9:   FROM read_parquet('yelp_parquet\review\year=*\data-*.parquet')
               ^

## 4) Časový split (train/valid/test)

In [ ]:

import pandas as pd

def temporal_split(parquet_path: Path, valid_ratio=0.1, test_ratio=0.1):
    df = pd.read_parquet(parquet_path, columns=["user_id","business_id","ts","stars","implicit"])
    df = df.dropna(subset=["user_id","business_id","ts"]).sort_values("ts")
    tmin, tmax = int(df.ts.min()), int(df.ts.max())
    span = tmax - tmin
    valid_cut = tmin + int((1 - (valid_ratio + test_ratio)) * span)
    test_cut  = tmin + int((1 - test_ratio) * span)
    train = df[df.ts <= valid_cut]
    valid = df[(df.ts > valid_cut) & (df.ts <= test_cut)]
    test  = df[df.ts > test_cut]
    return train, valid, test

train, valid, test = temporal_split(interactions_path, 0.1, 0.1)
[len(train), len(valid), len(test)], [train.ts.min(), train.ts.max(), test.ts.min(), test.ts.max()]


FileNotFoundError: [Errno 2] No such file or directory: 'Yelp JSON\\yelp_parquet\\interactions.parquet'

## 5) Příprava uživatelského a item indexu (ID mapping)

In [ ]:

# Mapování textových ID na integer indexy pro matice
def build_id_map(series):
    uniq = series.drop_duplicates().reset_index(drop=True)
    return pd.Series(index=uniq.values, data=np.arange(len(uniq)), name="idx")

uid_map = build_id_map(train["user_id"])
iid_map = build_id_map(train["business_id"])

def remap(df):
    df = df.copy()
    df["u"] = df["user_id"].map(uid_map)
    df["i"] = df["business_id"].map(iid_map)
    return df.dropna(subset=["u","i"]).astype({"u": int, "i": int})

train_m = remap(train); valid_m = remap(valid); test_m = remap(test)
train_m.head()


## 6) ELSA + SAE skeleton

In [ ]:

# --- ELSA skeleton ---
# ELSA je lineární model pracující s item reprezentacemi a podobnostmi.
# Zde necháme placeholder rozhraní; implementaci/volbu knihovny doplníme dle tvé preference.

class ELSAModel:
    def __init__(self, n_items: int, reg: float = 1e-4):
        self.n_items = n_items
        self.reg = reg
        self.item_factors = None  # např. R^k pro itemy (může vzniknout z EASE/ALS/…)

    def fit(self, interactions_df):
        # TODO: napojit na zvolený pretraining (EASE/ALS) → získat item_factors
        # a vytvořit podobnostní strukturu pro rychlé dotazy.
        pass

    def user_embed(self, user_history_indices):
        # TODO: složit embed z historie (agregace/federated attention)
        pass

    def recommend(self, user_history_indices, topk=50, filter_seen=True):
        # TODO: skórování přes podobnosti
        pass

# --- SAE skeleton (Top-K SAE) ---
# SAE bude trénovat nad uživatelskými nebo item embeddingy z ELSA a poskytne sparse interpretable neurony.
class TopKSAE:
    def __init__(self, input_dim: int, hidden_dim: int, k: int = 32, l1: float = 1e-3):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.k = k
        self.l1 = l1
        # TODO: weights init

    def fit(self, X):
        # TODO: trénink se sparsitou (Top-K aktivace / thresholding)
        pass

    def encode(self, X):
        # TODO: návrat sparse kódů
        pass



## 7) Další kroky
- Přidat filtry (země/region, roky) do `COPY … PARTITION_BY` pro menší subsety na rychlé iterace.
- Rozhodnout o pretrainingu pro ELSA (EASE/ALS/Item2Vec) a napojit `item_factors`.
- Přidat metriky (Recall@K, NDCG@K) a offline evaluaci na `valid/test`.
- Přidat export **explainers** z SAE (pojmenování neuronů ze slov v kategoriích/business attributes).



## Robustní globy + kontrola stromu (fix parquet hledání)
Níže se nastaví robustní patterny pro DuckDB/Hive particionaci a zároveň se vypíše strom Parquet souborů.
Pokud se liší, upravte `business_glob`/`review_glob` podle výpisu.


In [2]:

from pathlib import Path
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Pokud ještě nejsou, očekáváme proměnné:
PARQUET_DIR = Path("./yelp_parquet")
con = duckdb.connect("yelp.duckdb")

# Robustní globy pro Hive-like particionaci:
business_glob = (PARQUET_DIR / "business" / "state=*/*.parquet").as_posix()
review_glob   = (PARQUET_DIR / "review" / "year=*/*.parquet").as_posix()

def list_tree(root: Path, max_entries=200):
    rows = []
    if not root.exists():
        return rows
    for p in root.rglob("*.parquet"):
        try:
            rows.append(str(p.relative_to(root)))
        except Exception:
            rows.append(str(p))
        if len(rows) >= max_entries:
            break
    return rows

print("BUSINESS files:")
print("\n".join(list_tree(PARQUET_DIR / "business")) or "(nic)")
print("\n---\nREVIEW files:")
print("\n".join(list_tree(PARQUET_DIR / "review")) or "(nic)")
print("\n---\nUSER file:")
print([str(p) for p in (PARQUET_DIR).glob("user.parquet")] or "(nic)")

business_glob, review_glob


BUSINESS files:
state=AB\data_0.parquet
state=AZ\data_0.parquet
state=CA\data_0.parquet
state=CO\data_0.parquet
state=DE\data_0.parquet
state=FL\data_0.parquet
state=HI\data_0.parquet
state=ID\data_0.parquet
state=IL\data_0.parquet
state=IN\data_0.parquet
state=LA\data_0.parquet
state=MA\data_0.parquet
state=MI\data_0.parquet
state=MO\data_0.parquet
state=MT\data_0.parquet
state=NC\data_0.parquet
state=NJ\data_0.parquet
state=NV\data_0.parquet
state=PA\data_0.parquet
state=SD\data_0.parquet
state=TN\data_0.parquet
state=TX\data_0.parquet
state=UT\data_0.parquet
state=VI\data_0.parquet
state=VT\data_0.parquet
state=WA\data_0.parquet
state=XMS\data_0.parquet

---
REVIEW files:
year=2005\data_0.parquet
year=2006\data_0.parquet
year=2007\data_0.parquet
year=2008\data_0.parquet
year=2009\data_0.parquet
year=2010\data_0.parquet
year=2011\data_0.parquet
year=2012\data_0.parquet
year=2013\data_0.parquet
year=2014\data_0.parquet
year=2015\data_0.parquet
year=2016\data_0.parquet
year=2017\data_0

('yelp_parquet/business/state=*/*.parquet',
 'yelp_parquet/review/year=*/*.parquet')


## EDA grafy
Rychlé grafy: rozdělení hvězdiček podniků, počet recenzí ročně a rozdělení počtu recenzí na uživatele.


In [ ]:

# 1) Histogram hvězdiček podniků
stars_df = con.execute(f"""
SELECT CAST(stars AS DOUBLE) AS stars FROM read_parquet('{business_glob}')
WHERE stars IS NOT NULL
""").fetchdf()

plt.figure()
stars_df['stars'].plot(kind='hist', bins=20, title='Business stars (sample)')
plt.xlabel('stars'); plt.ylabel('count')
plt.show()


In [ ]:

# 2) Recenze za rok
per_year = con.execute(f"""
SELECT CAST(strftime(date, '%Y') AS INTEGER) AS y, COUNT(*) AS n
FROM read_parquet('{review_glob}')
GROUP BY y
ORDER BY y
""").fetchdf()

plt.figure()
plt.plot(per_year['y'], per_year['n'])
plt.title('Reviews per year')
plt.xlabel('year'); plt.ylabel('count')
plt.show()


In [ ]:

# 3) Aktivita uživatelů (počet recenzí na uživatele, cap na vzorku)
user_activity = con.execute(f"""
SELECT user_id, COUNT(*) AS n
FROM read_parquet('{review_glob}')
GROUP BY user_id
HAVING n >= 1
ORDER BY n DESC
LIMIT 200000
""").fetchdf()

plt.figure()
user_activity['n'].plot(kind='hist', bins=50, title='Reviews per user (sample, capped)')
plt.xlabel('reviews per user'); plt.ylabel('count')
plt.show()



## Interactions s pozitivním prahem + CSR matice
Implicitní interakce bereme jako recenze s `stars >= POS_THRESHOLD` (případně vše, pokud `USE_ANY_REVIEW=True`).
Uloží se také mapy ID a `.npz` s maticí.


In [ ]:

from scipy.sparse import coo_matrix, save_npz
import pickle

# Konfigurace implicit signálu
POS_THRESHOLD = 4.0
USE_ANY_REVIEW = False

# 1) Vyrobit interactions.parquet s prahem
interactions_path = PARQUET_DIR / "interactions.parquet"
if USE_ANY_REVIEW:
    where_clause = "user_id IS NOT NULL AND business_id IS NOT NULL"
else:
    where_clause = f"user_id IS NOT NULL AND business_id IS NOT NULL AND TRY_CAST(stars AS DOUBLE) >= {POS_THRESHOLD}"

con.execute(f"""
COPY (
  SELECT
    user_id,
    business_id,
    TRY_CAST(stars AS DOUBLE) AS stars,
    epoch_ms(CAST(date AS TIMESTAMP)) AS ts,
    1 AS implicit
  FROM read_parquet('{review_glob}')
  WHERE {where_clause}
) TO '{interactions_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD);
""")
print("interactions.parquet:", interactions_path.exists())

# 2) Vyrobit CSR a mapy ID
df = pd.read_parquet(interactions_path, columns=["user_id","business_id","ts","implicit"])
df = df.dropna(subset=["user_id","business_id"])

def build_id_map(series: pd.Series):
    uniq = series.drop_duplicates().reset_index(drop=True)
    return pd.Series(index=uniq.values, data=np.arange(len(uniq)), name="idx")

uid_map = build_id_map(df["user_id"])
iid_map = build_id_map(df["business_id"])

# Uložit mapy
with (PARQUET_DIR / "user2index.pkl").open("wb") as f: pickle.dump(uid_map.to_dict(), f)
with (PARQUET_DIR / "item2index.pkl").open("wb") as f: pickle.dump(iid_map.to_dict(), f)

df["u"] = df["user_id"].map(uid_map).astype(int)
df["i"] = df["business_id"].map(iid_map).astype(int)

n_users = int(df["u"].max()) + 1
n_items = int(df["i"].max()) + 1
vals = np.ones(len(df), dtype=np.float32)
R = coo_matrix((vals, (df["u"].values, df["i"].values)), shape=(n_users, n_items)).tocsr()

out_npz = PARQUET_DIR / "R_train.npz"
save_npz(out_npz, R)
print("CSR shape:", R.shape, "nnz:", R.nnz)
print("Saved:", out_npz, PARQUET_DIR / "user2index.pkl", PARQUET_DIR / "item2index.pkl")
